In [ ]:
import math
from datetime import timedelta

import polars as pl
import polars_distance as pld
import polars_h3 as plh3
import yaml
from haversine import haversine

# https://ion-elgreco.github.io/polars-distance/api/dist/#polars_distance.DistancePairWise.haversine

with open("../config.budapest.yaml") as fp:
    config = yaml.safe_load(fp)

In [150]:
def rg(data: pl.DataFrame) -> pl.DataFrame:
    d = data.with_columns(
        pl.struct("longitude_mean", "latitude_mean")
        .alias("stop")
        .struct.rename_fields(["longitude", "latitude"]),
        pl.struct(pl.col.longitude_mean.mean(), pl.col.latitude_mean.mean())
        .alias("centroid")
        .struct.rename_fields(["longitude", "latitude"]),
    ).with_columns(
        pld.col("centroid").dist.haversine("stop", "km").alias("haversine"),
        pl.col.interval_lenght.dt.total_seconds().sum().alias("N"),
    )
    res = (
        ((d["haversine"] ** 2 * d["interval_lenght"].dt.total_seconds()).sum() / d["N"])
        .sqrt()
        .round(3)[0]
    )
    return data.head(1).with_columns(pl.lit(res).alias("result"))

In [2]:
DIR = f"{config['dirs']['output']}/{config['city']}_ids_not_replaced"

In [3]:
for i in range(1):
    temp = pl.read_parquet(f"{DIR}/stops/v2/{i:02x}.parquet")

In [ ]:
test = temp.filter(pl.col.device_id.str.starts_with("0000051fc26276495c5a9ab1753a70"))[
    :3
].clone()

In [147]:
test = pl.DataFrame(
    {
        "device_id": [1, 1, 1, 1, 1],
        "place_label": [1, 1, 2, 2, 2],
        "interval_lenght": [
            timedelta(minutes=20),
            timedelta(minutes=30),
            timedelta(minutes=25),
            timedelta(minutes=15),
            timedelta(minutes=10),
        ],
        "latitude_mean": [
            47.6612,
            47.6612,
            47.4984,
            47.4984,
            47.4984,
        ],
        "longitude_mean": [19.0764, 19.0764, 19.0404, 19.0404, 19.0404],
    }
)

In [ ]:
c_lat = test["latitude_mean"].mean()
c_lon = test["longitude_mean"].mean()
p1 = haversine((c_lat, c_lon), (47.6612, 19.0764))
p2 = haversine((c_lat, c_lon), (47.4984, 19.0404))

In [ ]:
round(
    math.sqrt(
        sum([i**2 * n for i, n in zip([p1, p1, p2, p2, p2], [20, 30, 25, 15, 10])])
        / 100
    ),
    3,
)

9.333

In [130]:
round(math.sqrt(sum([i**2 * n for i, n in zip([p1, p2], [2, 3])]) / 5), 3)

8.967

In [205]:
test.group_by("device_id").map_groups(rg)
test.group_by("device_id").map_groups(rg_place)

device_id,result
i64,f64
1,8.967


In [143]:
for _ in range(1000):
    test.group_by("device_id").map_groups(rg)

In [204]:
def rg_place(data: pl.DataFrame) -> pl.DataFrame:
    d = data.with_columns(
        pl.struct("longitude_mean", "latitude_mean")
        .alias("stop")
        .struct.rename_fields(["longitude", "latitude"]),
        pl.struct(pl.col.longitude_mean.mean(), pl.col.latitude_mean.mean())
        .alias("centroid")
        .struct.rename_fields(["longitude", "latitude"]),
    ).with_columns(
        pld.col("centroid").dist.haversine("stop", "km").alias("haversine"),
    )
    place_counts = d.group_by("place_label").len()
    N = place_counts["len"].sum()
    d = d.join(place_counts, on="place_label", how="inner").unique(
        subset=("place_label")
    )
    res = round(math.sqrt((d["haversine"] ** 2 * d["len"]).sum() / N), 3)
    return pl.DataFrame({"device_id": [data["device_id"][0]], "result": [res]})

In [ ]:
q = rg(test)
q

8.241283721822473

In [50]:
((q["haversine"] ** 2 * q["interval_lenght"].dt.total_seconds()).sum() / q["N"]).sqrt()

N
f64
8.241284
8.241284
8.241284


In [ ]:
def rg_plh3(data: pl.DataFrame) -> pl.DataFrame:
    d = data.with_columns(
        pl.col.latitude_mean.mean().alias("c_lat"),
        pl.col.longitude_mean.mean().alias("c_lon"),
    ).with_columns(
        plh3.great_circle_distance(
            "latitude_mean",
            "longitude_mean",
            "c_lat",
            "c_lon",
            unit="km",
        ).alias("haversine"),
        pl.col.interval_lenght.dt.total_seconds().sum().alias("N"),
    )
    res = (
        ((d["haversine"] ** 2 * d["interval_lenght"].dt.total_seconds()).sum() / d["N"])
        .sqrt()
        .round(3)[0]
    )
    return data.head(1).with_columns(pl.lit(res).alias("result"))


def rg_hexagon(data: pl.DataFrame, res: int) -> pl.DataFrame:
    d = (
        data.with_columns(
            pl.col.latitude_mean.mean().alias("c_lat"),
            pl.col.longitude_mean.mean().alias("c_lon"),
        )
        .with_columns(
            plh3.int_to_str(
                plh3.latlng_to_cell(
                    pl.col("latitude_mean"), pl.col("longitude_mean"), 10
                )
            ).alias("h3_10"),
        )
        .with_columns(
            plh3.cell_to_latlng("h3_10").arr.get(0).alias("lat"),
            plh3.cell_to_latlng("h3_10").arr.get(1).alias("lon"),
        )
        .with_columns(
            plh3.great_circle_distance(
                "lat",
                "lon",
                "c_lat",
                "c_lon",
                unit="km",
            ).alias("haversine"),
            pl.col("h3_10").count().alias("N"),
        )
    )
    d.group_by("h3_10")
    res = (
        ((d["haversine"] ** 2 * d["h3_10"].dt.total_seconds()).sum() / d["N"])
        .sqrt()
        .round(3)[0]
    )
    return data.head(1).with_columns(pl.lit(res).alias("result"))


In [ ]:
temp.group_by("device_id").map_groups(rg_plh3)

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label,result
str,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64,f64
"""00a0ce86381dccd76065310d9e69b1…",1,2025-07-26 11:48:43 CEST,2025-07-26 12:42:32 CEST,53m 49s,47.629778,19.071876,2.0,1.0,8,1,4.542
"""00b5510a2b6a5c405062cc43e4eb97…",1,2024-02-26 09:35:48 CET,2024-02-26 09:46:14 CET,10m 26s,47.542504,19.259211,2.0,3.0,2,0,0.0
"""00a68d9a27a576ea19566f5660db36…",1,2025-07-11 00:04:24 CEST,2025-07-11 00:15:58 CEST,11m 34s,47.320522,19.075187,10.0,4.0,4,1,10.587
"""00c1b5f4f8b623e1f6ab73b4741f3f…",1,2025-10-24 14:51:21 CEST,2025-10-24 15:01:50 CEST,10m 29s,47.667335,19.645822,5.0,1.0,2,1,0.009
"""007c923f54c28e57b9b84e8309e3bd…",1,2025-03-09 09:13:54 CET,2025-03-09 09:19:24 CET,5m 30s,47.601712,19.111876,0.0,3.0,2,8,11.113
…,…,…,…,…,…,…,…,…,…,…,…
"""006ec30ade8d688d9ee95eefec6bc7…",1,2024-02-28 12:31:08 CET,2024-02-28 12:41:24 CET,10m 16s,47.476158,19.090433,6.0,5.0,2,4,15.771
"""008e1c7c7329158faec84014e8d846…",1,2025-07-17 15:44:11 CEST,2025-07-17 15:50:23 CEST,6m 12s,47.49518,19.023799,14.0,5.0,2,18,10.341
"""00704b34da0a9e916712a3471a7d15…",1,2025-06-22 22:07:08 CEST,2025-06-22 22:16:45 CEST,9m 37s,47.413524,19.094432,0.0,1.0,3,4,4.162


In [81]:
temp.group_by("device_id").map_groups(rg)


device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label,stop,centroid,haversine,N,result
str,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64,struct[2],struct[2],f64,i64,f64
"""00f70fce9f3bb2942e9238dfd9960b…",1,2023-10-27 02:15:10 CEST,2023-10-27 02:22:04 CEST,6m 54s,47.51374,19.219235,3.0,3.0,3,11,"{19.219235,47.51374}","{19.190824,47.527842}",2.647667,146016,14.244
"""0093107f97719ed33f6e48e030b01d…",1,2024-10-05 02:35:08 CEST,2024-10-05 02:53:35 CEST,18m 27s,47.441729,18.832365,9.0,14.0,4,1,"{18.832365,47.441729}","{18.830269,47.438448}",0.397394,2130,0.514
"""009694d5bbacb1ecb734bb62a9c854…",1,2025-12-23 16:09:02 CET,2025-12-23 16:16:22 CET,7m 20s,48.034006,19.9749,7.0,10.0,2,0,"{19.9749,48.034006}","{19.9749,48.034006}",0.0,440,0.0
"""004c98e5a49ac8a2d7e2515e51373c…",1,2025-08-28 15:19:15 CEST,2025-08-28 15:30:32 CEST,11m 17s,47.495732,19.064934,0.0,6.0,2,9,"{19.064934,47.495732}","{19.092882,47.507695}",2.48538,76444,2.132
"""00b8434096bfdc9ec58857a147bb7b…",1,2023-12-13 09:42:55 CET,2023-12-13 09:52:14 CET,9m 19s,47.441499,19.125149,3.0,1.0,4,3,"{19.125149,47.441499}","{19.128227,47.550818}",12.157867,629884,7.974
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""00599b7d2ee8d06336ca49c98ca5e4…",1,2025-06-22 20:20:57 CEST,2025-06-22 20:34:34 CEST,13m 37s,47.563829,19.095502,6.0,2.0,4,1,"{19.095502,47.563829}","{19.111834,47.49461}",7.793884,2195,5.67
"""00e88bd3e0fcd94d5e4aa5aeb046b3…",1,2023-11-05 09:44:29 CET,2023-11-05 09:50:38 CET,6m 9s,47.24001,18.63222,0.0,0.0,2,3,"{18.63222,47.24001}","{18.844284,47.367169}",21.345006,3951,18.971
"""001db4a1248ca9db6d1cf538f0fca2…",1,2025-07-15 09:18:09 CEST,2025-07-15 10:09:34 CEST,51m 25s,47.43995,19.169805,1.0,6.0,8,2,"{19.169805,47.43995}","{19.121671,47.510319}",8.620419,871653,4.303
